In [ ]:
# Fabric notebook parameters — injected by FabricSparkRouter via Jobs API
schema_path = ""
chunk_size = 500000
seed = 42
total_rows = 1000000
sinks_json = '{"sinks": ["lakehouse"], "sink_config": {}}'
workspace_id = ""
lakehouse_id = ""

In [ ]:
# Install sqllocks-spindle from GitHub main (Phase 1 + Phase 2 features)
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
     "sqllocks-spindle[fabric-files] @ git+https://github.com/sqllocks/spindle.git@main"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(f"pip install failed: {r.stderr}")
print("sqllocks-spindle installed from GitHub main")


In [ ]:
# Load augmented schema JSON from OneLake Files (absolute abfss URI)
import json
from notebookutils import mssparkutils

abfss_root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
schema_uri = f"{abfss_root}/Files/{schema_path}"
schema_raw = mssparkutils.fs.head(schema_uri, 200_000_000)
schema_dict = json.loads(schema_raw)
chunk_size = int(chunk_size)
seed = int(seed)
total_rows = int(total_rows)
sinks_config = json.loads(sinks_json)
sink_names = sinks_config.get("sinks", ["lakehouse"])
sink_cfg = sinks_config.get("sink_config", {})
print(f"Schema loaded. Static tables: {schema_dict.get('_static_tables', [])}")
print(f"Dynamic tables: {schema_dict.get('_dynamic_tables', [])}")


In [ ]:
# Distribute dynamic table generation across Spark partitions
import math
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

n_chunks = math.ceil(total_rows / chunk_size)

schema_dict_bc = sc.broadcast(schema_dict)
sink_names_bc = sc.broadcast(sink_names)
sink_cfg_bc = sc.broadcast(sink_cfg)
workspace_id_bc = sc.broadcast(workspace_id)
lakehouse_id_bc = sc.broadcast(lakehouse_id)
chunk_size_bc = sc.broadcast(chunk_size)
seed_bc = sc.broadcast(seed)
total_rows_bc = sc.broadcast(total_rows)

def generate_partition(chunk_indices):
    import json
    import os
    import tempfile
    import numpy as np
    from sqllocks_spindle.engine.chunk_worker import generate_chunk

    _schema = schema_dict_bc.value
    _chunk_size = chunk_size_bc.value
    _seed = seed_bc.value
    _total = total_rows_bc.value
    _sinks = sink_names_bc.value
    _sink_cfg = sink_cfg_bc.value
    _ws = workspace_id_bc.value
    _lh = lakehouse_id_bc.value

    with tempfile.NamedTemporaryFile(suffix=".json", mode="w", delete=False) as f:
        json.dump(_schema, f)
        tmp_path = f.name

    try:
        for i in chunk_indices:
            chunk_offset = i * _chunk_size
            chunk_count = min(_chunk_size, _total - chunk_offset)
            if chunk_count <= 0:
                continue
            chunk_seed = _seed ^ i

            chunk_data = generate_chunk(
                schema_path=tmp_path,
                seed=chunk_seed,
                offset=chunk_offset,
                count=chunk_count,
            )

            for sink_name in _sinks:
                if sink_name == "lakehouse":
                    from sqllocks_spindle.engine.sinks.lakehouse import LakehouseSink
                    abfss = f"abfss://{_ws}@onelake.dfs.fabric.microsoft.com/{_lh}"
                    sink = LakehouseSink(base_path=abfss, format="parquet")
                elif sink_name == "warehouse":
                    from sqllocks_spindle.engine.sinks.warehouse import WarehouseSink
                    sink = WarehouseSink(**_sink_cfg.get("warehouse", {}))
                elif sink_name == "kql":
                    from sqllocks_spindle.engine.sinks.kql import KQLSink
                    sink = KQLSink(**_sink_cfg.get("kql", {}))
                elif sink_name == "sql_database":
                    from sqllocks_spindle.engine.sinks.sql_database import SQLDatabaseSink
                    sink = SQLDatabaseSink(**_sink_cfg.get("sql_database", {}))
                else:
                    continue

                sink.open(None)
                for table_name, col_lists in chunk_data.items():
                    arrays = {col: np.array(vals) for col, vals in col_lists.items()}
                    sink.write_chunk(table_name, arrays)
                sink.close()
    finally:
        os.unlink(tmp_path)

print(f"Submitting {n_chunks} chunks across Spark partitions...")
sc.parallelize(range(n_chunks), numSlices=n_chunks).foreachPartition(generate_partition)
print("All partitions complete.")

In [ ]:
# Write result stats to OneLake and clean up temp schema file (absolute abfss URIs)
import json
from notebookutils import mssparkutils

result = {
    "rows_generated": total_rows,
    "status": "succeeded",
    "n_chunks": n_chunks,
}

abfss_root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
result_path = schema_path.replace("_schema.json", "_result.json")
result_uri = f"{abfss_root}/Files/{result_path}"
mssparkutils.fs.put(result_uri, json.dumps(result), overwrite=True)
print(f"Result written to OneLake: {result_uri}")

try:
    schema_uri = f"{abfss_root}/Files/{schema_path}"
    mssparkutils.fs.rm(schema_uri)
    print(f"Cleaned up schema file: {schema_uri}")
except Exception as e:
    print(f"Warning: could not delete schema file: {e}")
